In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:
data = sns.load_dataset("titanic")

In [ ]:
data.head()

In [ ]:
data.info()

In [ ]:
data.describe()

In [ ]:
data.size

In [ ]:
data_cleaned = data.copy()

# Removing Duplicate Rows

duplicate_count = data_cleaned.duplicated().sum()
print(f"Nummber of duplicate rows : {duplicate_count}")

duplicates = data_cleaned[data_cleaned.duplicated(keep=False)]
duplicates_sorted = duplicates.sort_values(by = duplicates.columns.tolist())
print(f"Dataframe Rows: {duplicates_sorted.shape[0]}")
print(f"Dataframe Columns : {duplicates_sorted.shape[1]}")

duplicates_sorted.head()



In [ ]:

initial_size = data_cleaned.size

data_cleaned = data_cleaned.drop_duplicates()

final_size = data_cleaned.size

proportion_removed = ((initial_size - final_size) / initial_size) * 100

print(f"Proportion of Data Removed: {proportion_removed:.2f}%")

# Handling Missing Data

In [ ]:
missing_data = data_cleaned.isnull().sum()

In [ ]:

missing_percentage = (missing_data / len(data_cleaned)) * 100

missing_df = pd.DataFrame({
    'Missing Values': missing_data,
    'Row Percentage': missing_percentage
})
missing_df

In [ ]:
plt.figure(figsize=(4, 4))
sns.heatmap(data_cleaned.isnull(), cbar=False);

In [ ]:
# Drop rows where the 'emarked' and 'embark_town' columns have missing values
data_cleaned = data_cleaned.dropna(subset=['embarked', 'embark_town'])

In [ ]:
# Dropping specific columns, e.g., dropping the 'deck'column
data_cleaned = data_cleaned.drop(columns=['deck'])

In [ ]:
# Filling missing data in 'age' column with mean values
data_cleaned['age'] = data_cleaned['age'].fillna(data_cleaned['age'].mean())

# Foward filling missing values
data_filled = data_cleaned.fillna(method='ffill')

In [ ]:
before = data.size
after = data_cleaned.size
difference = before - after
print(f"Proportion of data removed: {round((difference / before) *100, 2)}%")

data_cleaned.isnull().sum()

# Detecting and Managing Outliers

In [ ]:
plt.figure(figsize=(6, 1))
sns.boxplot(x=data_cleaned['fare']);

In [ ]:
plt.figure(figsize=(5,3))
sns.histplot(data_cleaned['fare'], bins=range(0,513,25));

In [ ]:
# Using a loop to identify outliers and determining the proportion
column_list = ['survived',
              'pclass',
              'age',
              'sibsp',
              'parch',
              'fare'
              ]

results = []

for column in column_list:
    q1 = data_cleaned[column].quantile(0.25)
    q3 = data_cleaned[column].quantile(0.75)
    iqr = q3 - q1

    # Standard threshold using Q3 + 1.5*IQR
    standard_threshold = q3 + 1.5 * iqr
    standard_outliers = (data_cleaned[column] > standard_threshold).sum()
    proportion = (standard_outliers / len(data_cleaned)) * 100

    results.append([
        column,
        standard_outliers,
        f"{proportion:.2f}%"
    ])

# Creating a DataFrame to display the results
outlier_df = pd.DataFrame(results, columns=[
    'Column',
    'Number of Outliers',
    'Proportion'
])

outlier_df

In [ ]:
# Defining lower and upper bounds to identify outliers
q1 = data_cleaned['parch'].quantile(0.25)
q3 = data_cleaned['parch'].quantile(0.75)
iqr = q3 - q1
lower_bound = q1 - 1.5 * iqr
upper_bound = q3 + 1.5 * iqr

# Remoing outliers by keeping only the rows within the IQR bounds
outliers_removed = data_cleaned[(data_cleaned['parch'] >= lower_bound) & (data_cleaned['parch'] <= upper_bound)]

# Checking the proportion of rows removed
proportion_removed = ((len(data_cleaned) - len(outliers_removed)) / len(data_cleaned)) * 100
print(f"Proportion Removed: {round(proportion_removed, 2)}%\n")

In [ ]:
from scipy import stats

In [ ]:
plt.figure(figsize=(5,3))
data_cleaned['age'].hist();

In [ ]:
mean = np.mean(data_cleaned['age'])
std_dev = np.std(data_cleaned['age'])

print(f"Mean: {mean}")
print(f"Standard Deviation: {std_dev}")

In [ ]:
# 1 standard deviation limits
lower_limit_1 = mean - std_dev
upper_limit_1 = mean + std_dev

# 2 standard deviation limits
lower_limit_2 = mean - 2 * std_dev
upper_limit_2 = mean + 2 * std_dev

# 3 standard deviation limits
lower_limit_3 = mean - 3 * std_dev
upper_limit_3 = mean + 3 * std_dev

In [ ]:
within_1_std = ((data_cleaned['age'] >= lower_limit_1) & (data_cleaned['age'] <= upper_limit_1)).mean()
within_2_std = ((data_cleaned['age'] >= lower_limit_2) & (data_cleaned['age'] <= upper_limit_2)).mean()
within_3_std = ((data_cleaned['age'] >= lower_limit_3) & (data_cleaned['age'] <= upper_limit_3)).mean()

print(f"{round(within_1_std * 100, 2)}% of data falls within 1 standard deviation.")
print(f"{round(within_2_std * 100, 2)}% of data falls within 2 standard deviations.")
print(f"{round(within_3_std * 100, 2)}% of data falls within 3 standard deviations.")

In [ ]:
data_cleaned['z_score'] = stats.zscore(data_cleaned['age'], ddof=1)
sample_df = data_cleaned[['age', 'z_score']]
sample_df.head()

In [ ]:
sample_df[(sample_df['z_score'] > 3) | (sample_df['z_score'] < -3)]

In [ ]:
outliers_removed = data_cleaned[(data_cleaned['z_score'] <= 3) & (data_cleaned['z_score'] >= -3)]

# Check the proportion of rows removed
proportion_removed = ((len(data_cleaned) - len(outliers_removed)) / len(data_cleaned)) * 100
print(f"Proportion Removed: {round(proportion_removed, 2)}%\n")

# Correcting Data Types

In [ ]:
data_cleaned.dtypes

In [ ]:
data_cleaned['embarked'] = data_cleaned['embarked'].astype('category')

In [ ]:
# Example of converting a date string column to datetime
data_cleaned['date_column'] = pd.to_datetime(data_cleaned['date_column'], errors='coerce')

# Summary
    1: Clear the duplicate cells
   
    2: Fill out the missing data cells by using mean, median.
   
    3: Remove the extrema that may reduce the model accuracy in the future

    4: Correct the data types of necessary columns